<a href="https://colab.research.google.com/github/e22171-lab/Statistical-Learning-e22171/blob/main/Bayesian_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


# Bayesian Estimation of a User Ability Parameter from Item Responses

For item \(i\), the response is

$$
Y_i=
\begin{cases}
1, & \text{if the answer is correct},\\
0, & \text{if the answer is incorrect}.
\end{cases}
$$

The two-parameter logistic item-response model is

$$
p_i(\theta)
=
P(Y_i=1\mid \Theta=\theta)
=
\frac{1}{1+e^{-a_i(\theta-b_i)}},
$$

where:

- \(a_i>0\) is the discrimination parameter.
- \(b_i\) is the difficulty parameter.
- \(\theta\) is the user's latent ability.

The initial prior distribution is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right),
$$

so that

$$
\Theta\sim N(0,1).
$$

---

# 1. Visualizing the Item-Response Curves

The response probability is

$$
p_i(\theta)
=
\frac{1}{1+e^{-a_i(\theta-b_i)}}.
$$

The difficulty \(b_i\) determines the horizontal position of the curve.

At

$$
\theta=b_i,
$$

we have

$$
p_i(\theta)=\frac{1}{2}.
$$

Therefore:

- Increasing \(b_i\) shifts the curve to the right.
- Decreasing \(b_i\) shifts the curve to the left.
- A large \(b_i\) represents a more difficult item.
- A small \(b_i\) represents an easier item.

The discrimination parameter \(a_i\) controls the steepness of the curve:

- A large \(a_i\) produces a steep curve.
- A small \(a_i\) produces a flatter curve.

In [1]:
import numpy as np
import plotly.graph_objects as go

def response_probability(theta, a, b):
    """2PL item-response probability."""
    z = np.clip(a * (theta - b), -700, 700)
    return 1 / (1 + np.exp(-z))

theta = np.linspace(-4, 4, 800)

# Two distinct discrimination values are used.
# a = 2.0 is paired with three different difficulty values.
curve_settings = [
    (0.7, 0.0, "a = 0.7, b = 0"),
    (2.0, -1.0, "a = 2.0, b = -1"),
    (2.0, 0.0, "a = 2.0, b = 0"),
    (2.0, 1.0, "a = 2.0, b = 1")
]

fig = go.Figure()

for a, b, label in curve_settings:
    probabilities = response_probability(theta, a, b)

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=probabilities,
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Two-Parameter Logistic Item-Response Curves",
    xaxis_title="Ability θ",
    yaxis_title="P(Y = 1 | θ)",
    template="plotly_white",
    yaxis=dict(range=[0, 1])
)

fig.show()

# 2. Sequential Likelihood Contribution

For a new response \(y_k\in\{0,1\}\), define

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

Because the response is Bernoulli, the likelihood contribution of the new response is

$$
\boxed{
L(y_k\mid\theta)
=
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{1-y_k}
}
$$

If the answer is correct, \(y_k=1\), then

$$
L(1\mid\theta)=p_k(\theta).
$$

If the answer is incorrect, \(y_k=0\), then

$$
L(0\mid\theta)=1-p_k(\theta).
$$

Assuming that responses are conditionally independent given \(\theta\), the joint likelihood for the response history

$$
\mathbf{y}^{(k)}=(y_1,y_2,\ldots,y_k)
$$

is

$$
\boxed{
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
\left[1-p_i(\theta)\right]^{1-y_i}
}
$$

---

# 3. Recursive Bayesian Posterior Update

At step \(k-1\), suppose the current posterior is

$$
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta\mid\mathbf{y}^{(k-1)}
\right).
$$

When the new response \(y_k\) is observed, Bayes' theorem gives

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta\mid\mathbf{y}^{(k)}
\right)
\propto
L(y_k\mid\theta)
f_{\Theta\mid\mathbf{Y}^{(k-1)}}
\left(
\theta\mid\mathbf{y}^{(k-1)}
\right).
$$

Therefore,

$$
\boxed{
f_k(\theta)
\propto
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{1-y_k}
f_{k-1}(\theta)
}
$$

The fully normalized posterior is

$$
\boxed{
f_k(\theta)
=
\frac{
L(y_k\mid\theta)f_{k-1}(\theta)
}{
\displaystyle
\int_{-\infty}^{\infty}
L(y_k\mid u)f_{k-1}(u)\,du
}
}
$$

The posterior after step \(k-1\) therefore becomes the prior for step \(k\).

---

# 4. Effect of a Correct Answer to a Difficult Item

Suppose the user correctly answers a difficult item:

$$
y_k=1
$$

with a large difficulty value \(b_k\).

The update becomes

$$
f_k(\theta)
\propto
p_k(\theta)f_{k-1}(\theta).
$$

Because

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}
$$

is an increasing function of \(\theta\), higher ability values receive greater likelihood.

For ability values much smaller than \(b_k\),

$$
p_k(\theta)\approx 0.
$$

Therefore, low values of \(\theta\) are strongly reduced in the posterior.

For ability values near or above \(b_k\),

$$
p_k(\theta)
$$

is much larger, so these values receive more posterior weight.

Mathematically,

$$
\frac{\partial}{\partial\theta}\ln p_k(\theta)
=
a_k\left[1-p_k(\theta)\right]>0.
$$

Thus, multiplying the previous posterior by \(p_k(\theta)\) adds positive slope to its log-density.

Consequently, the posterior peak generally shifts toward a larger value of \(\theta\):

$$
\boxed{
\text{Correct answer to a difficult item}
\quad\Longrightarrow\quad
\text{posterior shifts toward higher ability.}
}
$$

The shift is usually larger when the response was unlikely under the previous ability estimate.

---

# 5. Effect of the Discrimination Parameter

The discrimination parameter \(a_k\) controls how rapidly the response probability changes near

$$
\theta=b_k.
$$

The information supplied by item \(k\) at ability \(\theta\) is

$$
\boxed{
I_k(\theta)
=
a_k^2p_k(\theta)\left[1-p_k(\theta)\right]
}
$$

This is also related to the curvature of the log-likelihood:

$$
-\frac{\partial^2}{\partial\theta^2}
\ln L(y_k\mid\theta)
=
a_k^2p_k(\theta)\left[1-p_k(\theta)\right].
$$

## When \(a_k\) is large

A large \(a_k\):

- Produces a steep response curve.
- Strongly separates ability values below and above \(b_k\).
- Produces a sharper likelihood.
- Can reduce posterior variance substantially.
- Can cause a large change in the posterior after one response.

The item is most informative when the user's ability is close to its difficulty:

$$
\theta\approx b_k.
$$

However, when \(\theta\) is very far from \(b_k\), the probability is close to \(0\) or \(1\), and even a highly discriminating item may provide little additional information.

## When \(a_k\) is small

A small \(a_k\):

- Produces a flat response curve.
- Gives similar probabilities over many ability values.
- Produces a weak likelihood contribution.
- Causes only a small posterior update.
- Produces little reduction in posterior variance.

Therefore,

$$
\boxed{
\text{Large }a_k
\Rightarrow
\text{stronger and sharper update}
}
$$

while

$$
\boxed{
\text{Small }a_k
\Rightarrow
\text{weaker and flatter update}.
}
$$

---

# 6. Numerical Running-Grid Approximation

A numerical approximation can be maintained on a fixed grid

$$
\theta_1,\theta_2,\ldots,\theta_M.
$$

For example, use

$$
-4\leq\theta\leq4.
$$

## Step 1: Initialize the prior

Evaluate the standard normal prior at every grid point:

$$
f_0(\theta_m)
=
\frac{1}{\sqrt{2\pi}}
e^{-\theta_m^2/2}.
$$

Normalize it numerically.

## Step 2: Calculate response probabilities

For item \(k\), calculate

$$
p_{km}
=
\frac{1}{1+e^{-a_k(\theta_m-b_k)}}.
$$

## Step 3: Calculate the likelihood

For observed response \(y_k\),

$$
L_{km}
=
p_{km}^{y_k}
(1-p_{km})^{1-y_k}.
$$

## Step 4: Form the unnormalized posterior

Multiply the previous posterior by the new likelihood:

$$
\widetilde f_k(\theta_m)
=
L_{km}f_{k-1}(\theta_m).
$$

## Step 5: Sequential normalization

Approximate the normalization constant by numerical integration:

$$
C_k
\approx
\sum_{m=1}^{M}
\widetilde f_k(\theta_m)\Delta\theta.
$$

Then normalize:

$$
\boxed{
f_k(\theta_m)
=
\frac{\widetilde f_k(\theta_m)}{C_k}
}
$$

In Python, this can be performed using

```python
normalizing_constant = np.trapezoid(
    unnormalized_posterior,
    theta_grid
)

posterior = (
    unnormalized_posterior
    / normalizing_constant
)


### Colab Code Cell 2 — Running simulation for 20 items

```python
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------
# 1. Simulation settings
# ---------------------------------------------------

rng = np.random.default_rng(2026)

theta_true = 0.75
n_items = 20

theta_grid = np.linspace(-4, 4, 4001)

def sigmoid(z):
    """Numerically stable sigmoid function."""
    z = np.clip(z, -700, 700)
    return 1 / (1 + np.exp(-z))

def item_probability(theta, a, b):
    """2PL probability of a correct response."""
    return sigmoid(a * (theta - b))

# ---------------------------------------------------
# 2. Initial standard normal prior
# ---------------------------------------------------

posterior = (
    np.exp(-0.5 * theta_grid**2)
    / np.sqrt(2 * np.pi)
)

# Numerically normalize the initial prior
posterior /= np.trapezoid(
    posterior,
    theta_grid
)

# Store estimates beginning at step 0
steps = [0]

posterior_means = [
    np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )
]

map_estimates = [
    theta_grid[np.argmax(posterior)]
]

posterior_variances = [
    np.trapezoid(
        (theta_grid - posterior_means[0])**2
        * posterior,
        theta_grid
    )
]

item_records = []

# ---------------------------------------------------
# 3. Sequential Bayesian updates
# ---------------------------------------------------

for k in range(1, n_items + 1):

    # Random item parameters
    b_k = rng.normal(0, 1)
    a_k = rng.uniform(0.5, 2.0)

    # True probability of a correct answer
    p_true = item_probability(
        theta_true,
        a_k,
        b_k
    )

    # Simulate the response
    y_k = int(rng.random() < p_true)

    # Probability over the complete ability grid
    p_grid = item_probability(
        theta_grid,
        a_k,
        b_k
    )

    # Bernoulli likelihood
    likelihood = (
        p_grid**y_k
        * (1 - p_grid)**(1 - y_k)
    )

    # Unnormalized Bayesian update
    unnormalized_posterior = (
        posterior * likelihood
    )

    # Sequential normalization constant
    normalization_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if (
        not np.isfinite(normalization_constant)
        or normalization_constant <= 0
    ):
        raise ValueError(
            f"Invalid normalization constant at step {k}."
        )

    posterior = (
        unnormalized_posterior
        / normalization_constant
    )

    # Posterior mean
    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # MAP estimate
    map_estimate = theta_grid[
        np.argmax(posterior)
    ]

    # Posterior variance
    posterior_variance = np.trapezoid(
        (theta_grid - posterior_mean)**2
        * posterior,
        theta_grid
    )

    # Store the results
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)
    posterior_variances.append(
        posterior_variance
    )

    item_records.append({
        "Step": k,
        "Discrimination a": a_k,
        "Difficulty b": b_k,
        "True response probability": p_true,
        "Observed response y": y_k,
        "Posterior mean": posterior_mean,
        "MAP estimate": map_estimate,
        "Posterior SD": np.sqrt(
            posterior_variance
        )
    })

# ---------------------------------------------------
# 4. Display the running numerical results
# ---------------------------------------------------

results_table = pd.DataFrame(item_records)

display(
    results_table.round(4)
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior standard deviation:",
    round(
        np.sqrt(posterior_variances[-1]),
        4
    )
)

# ---------------------------------------------------
# 5. Plot the running estimators
# ---------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True ability θ = 0.75",
    annotation_position="top left"
)

fig.update_layout(
    title=(
        "Sequential Bayesian Estimation "
        "of User Ability"
    ),
    xaxis_title="Number of observed items",
    yaxis_title="Ability estimate",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# 7. Analysis of Convergence

The true ability used in the simulation is

$$
\theta_{\text{true}}=0.75.
$$

At step \(0\), the prior distribution is

$$
\Theta\sim N(0,1),
$$

so both the initial posterior mean and MAP estimate are approximately

$$
0.
$$

As more responses are observed, each likelihood contribution modifies the running posterior.

In general:

- Correct answers tend to move the ability estimate upward.
- Incorrect answers tend to move the ability estimate downward.
- Responses to highly discriminating items produce larger updates.
- Items with difficulty close to the user's true ability tend to provide more information.
- Items with very small discrimination provide relatively weak information.

The posterior mean and MAP estimate do not necessarily move toward the true value at every individual step. Their paths may fluctuate because:

1. Responses are randomly generated.
2. A user may occasionally answer an easy item incorrectly.
3. A user may occasionally answer a difficult item correctly.
4. Some items provide more information than others.

Nevertheless, as \(k\) increases, the cumulative likelihood generally becomes more concentrated around ability values that explain the complete response history.

Therefore, the posterior variance usually decreases:

$$
\operatorname{Var}
\left(
\Theta\mid\mathbf{Y}^{(k)}
\right)
\downarrow.
$$

A decreasing posterior variance means that the posterior becomes narrower and sharper. This indicates that the platform is becoming more confident about its estimate of the user's ability.

Thus,

$$
\boxed{
\text{More informative responses}
\Rightarrow
\text{posterior concentration}
\Rightarrow
\text{greater measurement confidence}.
}
$$

Because the simulation contains only \(20\) random items, the final posterior mean and MAP estimate may not be exactly equal to

$$
0.75.
$$

However, they should generally move closer to the true ability than their initial value, while the posterior uncertainty becomes smaller.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

# Bayesian Tracking of Click-Through Rates Using Beta–Bernoulli Updates

Let

$$
\Theta=\theta\in[0,1]
$$

represent the unknown true click-through rate of an advertisement.

For impression \(k\),

$$
Y_k=
\begin{cases}
1, & \text{if the user clicks},\\
0, & \text{if the user does not click}.
\end{cases}
$$

Conditional on \(\Theta=\theta\),

$$
Y_k\mid\Theta=\theta\sim\operatorname{Bernoulli}(\theta).
$$

Therefore,

$$
P(Y_k=1\mid\Theta=\theta)=\theta
$$

and

$$
P(Y_k=0\mid\Theta=\theta)=1-\theta.
$$

The initial prior is

$$
\Theta\sim\operatorname{Beta}(\alpha_0,\beta_0),
$$

with probability density

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(\alpha_0,\beta_0)}
\theta^{\alpha_0-1}(1-\theta)^{\beta_0-1},
\qquad 0<\theta<1.
$$

Here,

$$
B(\alpha,\beta)
=
\int_0^1
\theta^{\alpha-1}(1-\theta)^{\beta-1}\,d\theta
$$

is the Beta function.

---

# 1. Structural Probability and Properties

For a Beta distribution,

$$
\Theta\sim\operatorname{Beta}(\alpha,\beta),
$$

the probability density is

$$
\boxed{
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}(1-\theta)^{\beta-1}
}
$$

for

$$
0<\theta<1.
$$

The mean is

$$
\boxed{
\mathbb{E}[\Theta]
=
\frac{\alpha}{\alpha+\beta}
}
$$

and the variance is

$$
\boxed{
\operatorname{Var}(\Theta)
=
\frac{\alpha\beta}
{(\alpha+\beta)^2(\alpha+\beta+1)}
}
$$

For the three requested cases:

### Uniform state: \(\alpha=1,\beta=1\)

$$
f(\theta)=1.
$$

Every value between \(0\) and \(1\) has equal density.

The mean is

$$
\frac{1}{1+1}=0.5.
$$

### Right-skewed state: \(\alpha=2,\beta=8\)

$$
\mathbb{E}[\Theta]
=
\frac{2}{2+8}
=
0.2.
$$

Most of the probability mass is near \(0\), with a tail extending toward \(1\). This represents a relatively low CTR.

### Left-skewed state: \(\alpha=8,\beta=2\)

$$
\mathbb{E}[\Theta]
=
\frac{8}{8+2}
=
0.8.
$$

Most of the probability mass is near \(1\), with a tail extending toward \(0\). This represents a relatively high CTR.

Therefore:

- Increasing \(\alpha\) relative to \(\beta\) moves the density toward \(1\).
- Increasing \(\beta\) relative to \(\alpha\) moves the density toward \(0\).
- Increasing both parameters makes the distribution more concentrated.

In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta as beta_distribution

theta = np.linspace(0.001, 0.999, 1000)

parameter_pairs = [
    (1, 1, "Uniform: Beta(1, 1)"),
    (2, 8, "Right-skewed: Beta(2, 8)"),
    (8, 2, "Left-skewed: Beta(8, 2)")
]

fig = go.Figure()

for alpha, beta, label in parameter_pairs:
    density = beta_distribution.pdf(
        theta,
        alpha,
        beta
    )

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=density,
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Probability Density Functions of Beta Distributions",
    xaxis_title="Click-through rate θ",
    yaxis_title="Probability density",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# 2. Sequential Likelihood and Joint History

For one observed response \(y_k\in\{0,1\}\), the Bernoulli likelihood is

$$
\boxed{
L(y_k\mid\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}
}
$$

If a click is observed,

$$
y_k=1,
$$

then

$$
L(1\mid\theta)=\theta.
$$

If a non-click is observed,

$$
y_k=0,
$$

then

$$
L(0\mid\theta)=1-\theta.
$$

Let the running response history be

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k).
$$

Assuming conditional independence, the joint likelihood is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}(1-\theta)^{1-y_i}.
$$

Let

$$
s_k=\sum_{i=1}^{k}y_i
$$

be the total number of clicks.

Then the number of non-clicks is

$$
k-s_k.
$$

Therefore, the joint likelihood simplifies to

$$
\boxed{
L(\mathbf{y}^{(k)}\mid\theta)
=
\theta^{s_k}(1-\theta)^{k-s_k}
}
$$

---

# 3. Closed-Form Analytical Posterior Update

Suppose the posterior after step \(k-1\) is

$$
\Theta\mid\mathbf{Y}^{(k-1)}
\sim
\operatorname{Beta}(\alpha_{k-1},\beta_{k-1}).
$$

Therefore,

$$
f_{k-1}(\theta)
\propto
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

After observing \(y_k\), Bayes' theorem gives

$$
f_k(\theta)
\propto
L(y_k\mid\theta)f_{k-1}(\theta).
$$

Substitute the likelihood and previous posterior:

$$
f_k(\theta)
\propto
\theta^{y_k}(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining powers gives

$$
f_k(\theta)
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

This is another Beta distribution. Therefore,

$$
\boxed{
\Theta\mid\mathbf{Y}^{(k)}
\sim
\operatorname{Beta}(\alpha_k,\beta_k)
}
$$

where

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+1-y_k.
}
$$

Thus:

- A click adds \(1\) to \(\alpha\).
- A non-click adds \(1\) to \(\beta\).

Starting from \(\operatorname{Beta}(\alpha_0,\beta_0)\), after \(k\) observations,

$$
\boxed{
\alpha_k
=
\alpha_0+\sum_{i=1}^{k}y_i
}
$$

and

$$
\boxed{
\beta_k
=
\beta_0+k-\sum_{i=1}^{k}y_i.
}
$$

The normalized posterior density is

$$
\boxed{
f_k(\theta)
=
\frac{1}{B(\alpha_k,\beta_k)}
\theta^{\alpha_k-1}
(1-\theta)^{\beta_k-1}.
}
$$

This proves Beta–Bernoulli conjugacy.

---

## Posterior Mean

The posterior mean at step \(k\) is

$$
\boxed{
\mathbb{E}
\left[
\Theta\mid\mathbf{Y}^{(k)}
\right]
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
}
$$

Using the initial parameters and observed data,

$$
\boxed{
\widehat{\theta}_{Bayes}^{(k)}
=
\frac{
\alpha_0+\sum_{i=1}^{k}y_i
}{
\alpha_0+\beta_0+k
}.
}
$$

The posterior mean is also the posterior predictive probability of a click at the next impression:

$$
P(Y_{k+1}=1\mid\mathbf{Y}^{(k)})
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

---

# 4. Dynamic Shifting Mechanics

## Effect of a click

When

$$
y_k=1,
$$

the update is

$$
\alpha_k=\alpha_{k-1}+1
$$

and

$$
\beta_k=\beta_{k-1}.
$$

The new posterior is

$$
f_k(\theta)
\propto
\theta f_{k-1}(\theta).
$$

Multiplying by \(\theta\) gives more weight to larger values of \(\theta\). Therefore, the density shifts toward \(1\).

Let

$$
N_{k-1}=\alpha_{k-1}+\beta_{k-1}.
$$

The old posterior mean is

$$
m_{k-1}
=
\frac{\alpha_{k-1}}{N_{k-1}}.
$$

After a click,

$$
m_k
=
\frac{\alpha_{k-1}+1}{N_{k-1}+1}.
$$

The change is

$$
m_k-m_{k-1}
=
\frac{\beta_{k-1}}
{N_{k-1}(N_{k-1}+1)}
>0.
$$

Therefore, a click increases the posterior mean.

---

## Effect of a non-click

When

$$
y_k=0,
$$

the update is

$$
\alpha_k=\alpha_{k-1}
$$

and

$$
\beta_k=\beta_{k-1}+1.
$$

The new posterior is

$$
f_k(\theta)
\propto
(1-\theta)f_{k-1}(\theta).
$$

Multiplying by \(1-\theta\) gives more weight to smaller values of \(\theta\). Therefore, the density shifts toward \(0\).

After a non-click,

$$
m_k
=
\frac{\alpha_{k-1}}{N_{k-1}+1}.
$$

The change is

$$
m_k-m_{k-1}
=
-\frac{\alpha_{k-1}}
{N_{k-1}(N_{k-1}+1)}
<0.
$$

Therefore, a non-click decreases the posterior mean.

Hence,

$$
\boxed{
\text{Click}\Rightarrow\text{posterior shifts toward }1
}
$$

and

$$
\boxed{
\text{Non-click}\Rightarrow\text{posterior shifts toward }0.
}
$$

---

## Comparison with a Non-Conjugate Model

For the Beta–Bernoulli model, the prior and posterior belong to the same Beta family.

The update only requires

$$
\alpha_k=\alpha_{k-1}+y_k
$$

and

$$
\beta_k=\beta_{k-1}+1-y_k.
$$

Therefore, no numerical integration is needed.

In a non-conjugate model such as the 2PL item-response model,

$$
P(Y_k=1\mid\theta)
=
\frac{1}
{1+\exp[-a_k(\theta-b_k)]},
$$

multiplying a normal prior by the logistic likelihood does not produce another normal distribution.

Therefore, the posterior usually requires numerical methods such as:

- Fixed-grid numerical integration
- Laplace approximation
- Markov Chain Monte Carlo
- Variational inference

Thus,

$$
\boxed{
\text{Beta–Bernoulli: exact closed-form update}
}
$$

while

$$
\boxed{
\text{2PL IRT: numerical posterior approximation is required}.
}
$$

---

# 5. Running Point Estimators

## Posterior Mean

The running Bayesian posterior mean is

$$
\boxed{
\widehat{\theta}_{Bayes}^{(k)}
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
}
$$

---

## Maximum A Posteriori Estimate

If

$$
\alpha_k>1
\quad\text{and}\quad
\beta_k>1,
$$

the Beta distribution has an interior mode:

$$
\boxed{
\widehat{\theta}_{MAP}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}.
}
$$

Boundary cases must be treated separately:

$$
\widehat{\theta}_{MAP}^{(k)}
=
\begin{cases}
\dfrac{\alpha_k-1}{\alpha_k+\beta_k-2},
& \alpha_k>1,\;\beta_k>1,\\[10pt]
0,
& \alpha_k\leq1,\;\beta_k>1,\\[4pt]
1,
& \alpha_k>1,\;\beta_k\leq1.
\end{cases}
$$

When

$$
\alpha_k=\beta_k=1,
$$

the distribution is uniform, so every value in \([0,1]\) is a mode. Therefore, the MAP is not unique.

For plotting, \(0.5\) may be used as a neutral representative of the uniform prior.

In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------------------------------------------------
# 1. Settings
# ---------------------------------------------------

rng = np.random.default_rng(2026)

theta_true = 0.35
n_impressions = 100

alpha = 1
beta = 1

def beta_map(alpha, beta):
    """
    Return the MAP estimate of a Beta distribution.

    For Beta(1,1), the mode is not unique.
    We return 0.5 only as a neutral plotting value.
    """

    if alpha > 1 and beta > 1:
        return (alpha - 1) / (alpha + beta - 2)

    if alpha > 1 and beta <= 1:
        return 1.0

    if alpha <= 1 and beta > 1:
        return 0.0

    if alpha == 1 and beta == 1:
        return 0.5

    # This case does not occur when starting from Beta(1,1)
    return np.nan

# ---------------------------------------------------
# 2. Store step-0 values
# ---------------------------------------------------

steps = [0]

posterior_means = [
    alpha / (alpha + beta)
]

map_estimates = [
    beta_map(alpha, beta)
]

posterior_variances = [
    (
        alpha * beta
        / (
            (alpha + beta)**2
            * (alpha + beta + 1)
        )
    )
]

records = [{
    "Step": 0,
    "Response": np.nan,
    "Alpha": alpha,
    "Beta": beta,
    "Posterior Mean": posterior_means[0],
    "MAP Estimate": map_estimates[0],
    "Posterior SD": np.sqrt(
        posterior_variances[0]
    )
}]

# ---------------------------------------------------
# 3. Sequential Beta–Bernoulli updates
# ---------------------------------------------------

for k in range(1, n_impressions + 1):

    # Simulate one response
    y_k = int(
        rng.random() < theta_true
    )

    # Closed-form update
    alpha = alpha + y_k
    beta = beta + (1 - y_k)

    # Posterior mean
    posterior_mean = (
        alpha / (alpha + beta)
    )

    # MAP estimate
    map_estimate = beta_map(
        alpha,
        beta
    )

    # Posterior variance
    posterior_variance = (
        alpha * beta
        / (
            (alpha + beta)**2
            * (alpha + beta + 1)
        )
    )

    steps.append(k)
    posterior_means.append(
        posterior_mean
    )
    map_estimates.append(
        map_estimate
    )
    posterior_variances.append(
        posterior_variance
    )

    records.append({
        "Step": k,
        "Response": y_k,
        "Alpha": alpha,
        "Beta": beta,
        "Posterior Mean": posterior_mean,
        "MAP Estimate": map_estimate,
        "Posterior SD": np.sqrt(
            posterior_variance
        )
    })

# ---------------------------------------------------
# 4. Display numerical results
# ---------------------------------------------------

results = pd.DataFrame(records)

display(
    results.tail(15).round(4)
)

print(
    "Total clicks:",
    int(results["Response"].sum())
)

print(
    "Final alpha:",
    alpha
)

print(
    "Final beta:",
    beta
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior standard deviation:",
    round(
        np.sqrt(posterior_variances[-1]),
        4
    )
)

# ---------------------------------------------------
# 5. Plot the estimators
# ---------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR θ = 0.35",
    annotation_position="top left"
)

fig.update_layout(
    title=(
        "Sequential Bayesian Tracking "
        "of Advertisement CTR"
    ),
    xaxis_title="Number of impressions",
    yaxis_title="Estimated click-through rate",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

,Step,Response,Alpha,Beta,Posterior Mean,MAP Estimate,Posterior SD
86,86,0.0,28,60,0.3182,0.3140,0.0494
87,87,0.0,28,61,0.3146,0.3103,0.0489
88,88,0.0,28,62,0.3111,0.3068,0.0485
89,89,0.0,28,63,0.3077,0.3034,0.0481
90,90,0.0,28,64,0.3043,0.3000,0.0477
91,91,0.0,28,65,0.3011,0.2967,0.0473
92,92,1.0,29,65,0.3085,0.3043,0.0474
93,93,0.0,29,66,0.3053,0.3011,0.0470
94,94,0.0,29,67,0.3021,0.2979,0.0466
95,95,0.0,29,68,0.2990,0.2947,0.0462


Total clicks: 29
Final alpha: 30
Final beta: 72
Final posterior mean: 0.2941
Final MAP estimate: 0.29
Final posterior standard deviation: 0.0449


# 6. Performance Tracking and Convergence Analysis

The true click-through rate used in the simulation is

$$
\theta_{\text{true}}=0.35.
$$

The initial prior is

$$
\Theta\sim\operatorname{Beta}(1,1),
$$

which is uniform over \([0,1]\).

Its initial posterior mean is

$$
\widehat{\theta}_{Bayes}^{(0)}
=
\frac{1}{1+1}
=
0.5.
$$

The uniform prior has no unique MAP because every value between \(0\) and \(1\) has equal density. In the plot, \(0.5\) is used only as a neutral representative.

After \(k\) observations, let \(s_k\) be the number of clicks. Then

$$
\alpha_k=1+s_k
$$

and

$$
\beta_k=1+k-s_k.
$$

Therefore, the posterior mean is

$$
\widehat{\theta}_{Bayes}^{(k)}
=
\frac{1+s_k}{k+2}.
$$

Once both clicks and non-clicks have been observed, the MAP estimate is

$$
\widehat{\theta}_{MAP}^{(k)}
=
\frac{s_k}{k}.
$$

Thus, with the uniform prior, the interior MAP estimate becomes equal to the sample click proportion.

As \(k\) increases, the observed click proportion generally approaches the true CTR:

$$
\frac{s_k}{k}
\longrightarrow
\theta_{\text{true}}.
$$

Therefore, both estimators generally move toward

$$
0.35.
$$

They may fluctuate at the beginning because the number of observations is small. A single click or non-click can then cause a relatively large change.

As more observations are collected:

- Each new response has a smaller effect.
- The estimates become more stable.
- The posterior becomes more concentrated.
- The influence of the initial prior decreases.
- The observed data dominate the posterior.

The posterior variance is

$$
\operatorname{Var}
\left(
\Theta\mid\mathbf{Y}^{(k)}
\right)
=
\frac{
\alpha_k\beta_k
}{
(\alpha_k+\beta_k)^2
(\alpha_k+\beta_k+1)
}.
$$

Since

$$
\alpha_k+\beta_k
=
\alpha_0+\beta_0+k,
$$

the denominator increases as \(k\) increases. Therefore, the posterior variance generally decreases.

This means that the posterior distribution becomes narrower and sharper around the estimated CTR.

Hence,

$$
\boxed{
\text{More impressions}
\Rightarrow
\text{more accumulated evidence}
\Rightarrow
\text{smaller uncertainty}.
}
$$

For a large sample, the exact choice of the initial prior becomes less important because the data contribution grows with \(k\), while the prior contributes only the fixed pseudo-counts

$$
\alpha_0
\quad\text{and}\quad
\beta_0.
$$

Therefore,

$$
\boxed{
\text{As }k\to\infty,\text{ the likelihood dominates the initial prior.}
}
$$

With only \(100\) simulated impressions, the final estimators may not equal \(0.35\) exactly because the responses are random. However, they should normally be closer to the true CTR than the initial value \(0.5\), and the posterior uncertainty should be substantially smaller.

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

# Bayesian Estimation for Structural Health Monitoring Using Bounded Grid Updates

Let

$$
\Theta=\theta
$$

represent the remaining stiffness efficiency factor of the structure, where

$$
0<\theta\leq 1.
$$

The physical interpretation is:

- \(\theta=1\): completely healthy structure.
- \(\theta\to 0\): severe degradation or structural damage.

Let \(K_{\text{nominal}}\) be the stiffness of the completely healthy structure.

At inspection step \(k\), the sensor model is

$$
Y_k=\theta K_{\text{nominal}}e^{\varepsilon_k},
$$

where

$$
\varepsilon_k\sim N(0,\sigma^2).
$$

Therefore, the measurement noise is multiplicative and log-normal.

---

# 1. Prior Belief Boundaries

The initial prior is

$$
\Theta\sim\operatorname{Beta}(8,1.5).
$$

The Beta probability density is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{8-1}(1-\theta)^{1.5-1},
\qquad 0<\theta<1.
$$

Therefore,

$$
\boxed{
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(8,1.5)}
\theta^7(1-\theta)^{0.5}
}
$$

The mean of a Beta distribution is

$$
\mathbb{E}[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Hence,

$$
\mathbb{E}[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}.
$$

Therefore,

$$
\boxed{
\mathbb{E}[\Theta^{(0)}]
\approx 0.8421
}
$$

The prior mode is

$$
\frac{\alpha-1}{\alpha+\beta-2}
=
\frac{8-1}{8+1.5-2}
=
\frac{7}{7.5}
\approx 0.9333.
$$

Thus, the prior places most of its probability mass near high stiffness values.

This is appropriate because engineers initially believe that the structure is likely to be healthy before any evidence of damage is observed.

However, the prior still assigns some probability to lower values of \(\theta\), allowing sensor measurements to provide evidence of degradation.

In [4]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta as beta_distribution

# Avoid theta = 0 because later calculations contain log(theta)
theta_grid = np.linspace(0.001, 1.0, 5000)

alpha_0 = 8.0
beta_0 = 1.5

prior_density = beta_distribution.pdf(
    theta_grid,
    alpha_0,
    beta_0
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5) prior"
    )
)

fig.update_layout(
    title="Initial Prior for Structural Stiffness Efficiency",
    xaxis_title="Remaining stiffness factor θ",
    yaxis_title="Probability density",
    template="plotly_white"
)

fig.show()

prior_mean = alpha_0 / (alpha_0 + beta_0)
prior_mode = (
    (alpha_0 - 1)
    / (alpha_0 + beta_0 - 2)
)

print("Prior mean:", round(prior_mean, 4))
print("Prior mode:", round(prior_mode, 4))

Prior mean: 0.8421
Prior mode: 0.9333


# 2. Structural Likelihood Formulation

The measurement model is

$$
Y_k=\theta K_{\text{nominal}}e^{\varepsilon_k},
$$

where

$$
\varepsilon_k\sim N(0,\sigma^2).
$$

Take logarithms:

$$
\ln Y_k
=
\ln(\theta K_{\text{nominal}})
+
\varepsilon_k.
$$

Therefore,

$$
\ln Y_k\mid\Theta=\theta
\sim
N\left(
\ln(\theta K_{\text{nominal}}),
\sigma^2
\right).
$$

Thus, conditional on \(\theta\), \(Y_k\) follows a log-normal distribution.

The likelihood contribution of a single sensor measurement \(y_k\) is

$$
\boxed{
L(y_k\mid\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\ln y_k-\ln(\theta K_{\text{nominal}})
\right)^2
}
{2\sigma^2}
\right]
}
$$

for

$$
y_k>0
\quad\text{and}\quad
0<\theta\leq1.
$$

Equivalently,

$$
\boxed{
L(y_k\mid\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left[
\ln\left(
\frac{y_k}
{\theta K_{\text{nominal}}}
\right)
\right]^2
}
{2\sigma^2}
\right].
}
$$

Let the running measurement history be

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k).
$$

Assuming conditional independence of the sensor measurements given \(\theta\), the joint likelihood is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
L(y_i\mid\theta).
$$

Therefore,

$$
\boxed{
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\frac{1}
{y_i\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left[
\ln\left(
\frac{y_i}
{\theta K_{\text{nominal}}}
\right)
\right]^2
}
{2\sigma^2}
\right]
}
$$

or

$$
\boxed{
L(\mathbf{y}^{(k)}\mid\theta)
=
\left[
\prod_{i=1}^{k}
\frac{1}
{y_i\sigma\sqrt{2\pi}}
\right]
\exp\left[
-\frac{1}{2\sigma^2}
\sum_{i=1}^{k}
\left[
\ln\left(
\frac{y_i}
{\theta K_{\text{nominal}}}
\right)
\right]^2
\right].
}
$$

The factors outside the exponential do not depend on \(\theta\), but they are part of the complete likelihood.

---

# 3. Non-Conjugate Posterior Update

The initial Beta prior has the form

$$
f_0(\theta)
\propto
\theta^{\alpha_0-1}
(1-\theta)^{\beta_0-1}.
$$

The log-normal likelihood contains

$$
\exp\left[
-\frac{
\left(
\ln y_k-\ln\theta-\ln K_{\text{nominal}}
\right)^2
}
{2\sigma^2}
\right].
$$

This expression includes nonlinear terms involving

$$
(\ln\theta)^2.
$$

Multiplying this likelihood by a Beta density does not produce another Beta distribution.

Therefore, the Beta prior is not conjugate to the log-normal structural likelihood.

Hence, the posterior does not have a simple closed-form named distribution.

At step \(k\), Bayes' theorem gives

$$
f_k(\theta)
=
f_{\Theta\mid\mathbf{Y}^{(k)}}
\left(
\theta\mid\mathbf{y}^{(k)}
\right).
$$

The recursive update is

$$
\boxed{
f_k(\theta)
\propto
L(y_k\mid\theta)f_{k-1}(\theta)
}
$$

or explicitly,

$$
\boxed{
f_k(\theta)
\propto
f_{k-1}(\theta)
\exp\left[
-\frac{
\left[
\ln\left(
\frac{y_k}
{\theta K_{\text{nominal}}}
\right)
\right]^2
}
{2\sigma^2}
\right].
}
$$

The normalized update is

$$
\boxed{
f_k(\theta)
=
\frac{
L(y_k\mid\theta)f_{k-1}(\theta)
}{
\displaystyle
\int_0^1
L(y_k\mid u)f_{k-1}(u)\,du
}.
}
$$

The denominator is the sequential normalization constant.

---

# 4. Running Point Estimates

Because the posterior does not have a closed-form distribution, the point estimates must be calculated numerically.

## Posterior Mean

The running Bayesian posterior mean is

$$
\boxed{
\widehat{\theta}_{Bayes}^{(k)}
=
\mathbb{E}
\left[
\Theta\mid\mathbf{Y}^{(k)}
\right]
=
\int_0^1
\theta f_k(\theta)\,d\theta.
}
$$

Because the posterior density is normalized,

$$
\int_0^1f_k(\theta)\,d\theta=1.
$$

If the posterior is stored only up to proportionality, the mean can be calculated as

$$
\boxed{
\widehat{\theta}_{Bayes}^{(k)}
=
\frac{
\displaystyle
\int_0^1
\theta\widetilde f_k(\theta)\,d\theta
}{
\displaystyle
\int_0^1
\widetilde f_k(\theta)\,d\theta
}.
}
$$

## Maximum A Posteriori Estimate

The MAP estimate is the value of \(\theta\) at which the posterior density is largest:

$$
\boxed{
\widehat{\theta}_{MAP}^{(k)}
=
\underset{0<\theta\leq1}
{\operatorname{argmax}}
\;
f_k(\theta).
}
$$

On a numerical grid, it is approximated by

$$
\boxed{
\widehat{\theta}_{MAP}^{(k)}
\approx
\theta_{m^*},
\qquad
m^*
=
\underset{m}{\operatorname{argmax}}
\;
f_k(\theta_m).
}
$$

The posterior variance is

$$
\boxed{
\operatorname{Var}
\left(
\Theta\mid\mathbf{Y}^{(k)}
\right)
=
\int_0^1
\left(
\theta-\widehat{\theta}_{Bayes}^{(k)}
\right)^2
f_k(\theta)\,d\theta.
}
$$

# 5. Algorithmic Grid Approximation and Normalization

Choose a dense numerical grid

$$
\theta_1,\theta_2,\ldots,\theta_M
$$

over the physical domain.

Because the likelihood contains \(\ln\theta\), the point

$$
\theta=0
$$

must not be included.

Therefore, use a small positive lower boundary such as

$$
\theta_{\min}=0.001
$$

and define

$$
\theta_m\in[0.001,1].
$$

## Step 1: Initialize the prior

Evaluate

$$
f_0(\theta_m)
=
\frac{1}{B(8,1.5)}
\theta_m^7(1-\theta_m)^{0.5}.
$$

Normalize numerically:

$$
f_0(\theta_m)
\leftarrow
\frac{
f_0(\theta_m)
}{
\displaystyle
\int_{0.001}^{1}
f_0(\theta)\,d\theta
}.
$$

Using the trapezoidal rule,

$$
\int_{0.001}^{1}
f_0(\theta)\,d\theta
\approx
\operatorname{trapezoid}
\left(
f_0(\theta_m),\theta_m
\right).
$$

## Step 2: Evaluate the new likelihood

After observing \(y_k\), calculate at every grid point

$$
L_{km}
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left[
\ln\left(
\frac{y_k}
{\theta_mK_{\text{nominal}}}
\right)
\right]^2
}
{2\sigma^2}
\right].
$$

## Step 3: Form the unnormalized posterior

Calculate

$$
\widetilde f_k(\theta_m)
=
L_{km}f_{k-1}(\theta_m).
$$

## Step 4: Calculate the normalization constant

Use the trapezoidal rule:

$$
C_k
\approx
\operatorname{trapezoid}
\left(
\widetilde f_k(\theta_m),
\theta_m
\right).
$$

In Python:

```python
normalization_constant = np.trapezoid(
    unnormalized_posterior,
    theta_grid
)


### Colab Code Cell 2 — Complete sequential simulation

```python
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import beta as beta_distribution

# ---------------------------------------------------
# 1. Model settings
# ---------------------------------------------------

rng = np.random.default_rng(2026)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_measurements = 15

alpha_0 = 8.0
beta_0 = 1.5

# Avoid theta = 0 because log(theta) is undefined
theta_grid = np.linspace(
    0.001,
    1.0,
    5000
)

# ---------------------------------------------------
# 2. Initial Beta prior
# ---------------------------------------------------

posterior = beta_distribution.pdf(
    theta_grid,
    alpha_0,
    beta_0
)

posterior /= np.trapezoid(
    posterior,
    theta_grid
)

def calculate_posterior_statistics(
    theta_values,
    density
):
    """Calculate posterior mean, MAP and standard deviation."""

    mean = np.trapezoid(
        theta_values * density,
        theta_values
    )

    map_estimate = theta_values[
        np.argmax(density)
    ]

    variance = np.trapezoid(
        (theta_values - mean) ** 2
        * density,
        theta_values
    )

    return (
        mean,
        map_estimate,
        np.sqrt(variance)
    )

prior_mean, prior_map, prior_sd = (
    calculate_posterior_statistics(
        theta_grid,
        posterior
    )
)

steps = [0]
posterior_means = [prior_mean]
map_estimates = [prior_map]
posterior_sds = [prior_sd]

milestone_steps = {0, 1, 2, 5, 10, 15}
posterior_milestones = {
    0: posterior.copy()
}

records = [{
    "Step": 0,
    "Sensor reading": np.nan,
    "Posterior mean": prior_mean,
    "MAP estimate": prior_map,
    "Posterior SD": prior_sd
}]

# ---------------------------------------------------
# 3. Sequential sensor measurements
# ---------------------------------------------------

for k in range(1, n_measurements + 1):

    # Generate log-normal multiplicative noise
    epsilon_k = rng.normal(
        loc=0.0,
        scale=sigma
    )

    # Simulate the sensor reading
    y_k = (
        theta_true
        * K_nominal
        * np.exp(epsilon_k)
    )

    # Evaluate the log-normal likelihood on the grid
    log_ratio = np.log(
        y_k
        / (theta_grid * K_nominal)
    )

    likelihood = (
        1
        / (
            y_k
            * sigma
            * np.sqrt(2 * np.pi)
        )
    ) * np.exp(
        -0.5
        * (log_ratio / sigma) ** 2
    )

    # Form the unnormalized posterior
    unnormalized_posterior = (
        posterior * likelihood
    )

    # Sequential normalization using trapezoidal integration
    normalization_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if (
        not np.isfinite(normalization_constant)
        or normalization_constant <= 0
    ):
        raise ValueError(
            f"Invalid normalization constant at step {k}."
        )

    posterior = (
        unnormalized_posterior
        / normalization_constant
    )

    # Calculate running estimates
    posterior_mean, map_estimate, posterior_sd = (
        calculate_posterior_statistics(
            theta_grid,
            posterior
        )
    )

    steps.append(k)
    posterior_means.append(
        posterior_mean
    )
    map_estimates.append(
        map_estimate
    )
    posterior_sds.append(
        posterior_sd
    )

    records.append({
        "Step": k,
        "Sensor reading": y_k,
        "Posterior mean": posterior_mean,
        "MAP estimate": map_estimate,
        "Posterior SD": posterior_sd
    })

    if k in milestone_steps:
        posterior_milestones[k] = (
            posterior.copy()
        )

# ---------------------------------------------------
# 4. Display the numerical results
# ---------------------------------------------------

results = pd.DataFrame(records)

display(
    results.round(4)
)

print(
    "Initial posterior mean:",
    round(posterior_means[0], 4)
)

print(
    "Final posterior mean:",
    round(posterior_means[-1], 4)
)

print(
    "Final MAP estimate:",
    round(map_estimates[-1], 4)
)

print(
    "Final posterior standard deviation:",
    round(posterior_sds[-1], 4)
)

# ---------------------------------------------------
# 5. Define an operational confidence criterion
# ---------------------------------------------------

# Example criterion:
# estimate is within 0.03 of the true stiffness
# and posterior SD is below 0.05

confidence_step = None

for k in range(1, n_measurements + 1):
    close_to_true = (
        abs(
            posterior_means[k]
            - theta_true
        )
        < 0.03
    )

    sufficiently_narrow = (
        posterior_sds[k]
        < 0.05
    )

    if close_to_true and sufficiently_narrow:
        confidence_step = k
        break

print(
    "First confidence step:",
    confidence_step
)

In [6]:
fig_density = go.Figure()

for k in sorted(posterior_milestones):
    fig_density.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_milestones[k],
            mode="lines",
            name=f"Step {k}"
        )
    )

fig_density.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True stiffness θ = 0.68",
    annotation_position="top"
)

fig_density.update_layout(
    title=(
        "Sequential Posterior Density "
        "for Remaining Structural Stiffness"
    ),
    xaxis_title="Remaining stiffness factor θ",
    yaxis_title="Posterior density",
    template="plotly_white",
    hovermode="x unified"
)

fig_density.show()

NameError: name 'posterior_milestones' is not defined

In [7]:
fig_estimates = go.Figure()

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior mean"
    )
)

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP estimate"
    )
)

fig_estimates.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True stiffness θ = 0.68",
    annotation_position="top left"
)

fig_estimates.update_layout(
    title=(
        "Convergence of Structural "
        "Stiffness Estimates"
    ),
    xaxis_title="Number of sensor measurements",
    yaxis_title="Estimated stiffness factor",
    yaxis=dict(range=[0, 1]),
    template="plotly_white",
    hovermode="x unified"
)

fig_estimates.show()

# 6. Performance Tracking and Degradation Analysis

The true remaining stiffness is

$$
\theta_{\text{true}}=0.68.
$$

The initial prior is

$$
\Theta\sim\operatorname{Beta}(8,1.5),
$$

with prior mean

$$
0.8421
$$

and prior mode approximately

$$
0.9333.
$$

Therefore, before receiving any sensor measurements, the system strongly expects the structure to be healthy.

After the impact, the sensor measurements are generated around

$$
\theta_{\text{true}}K_{\text{nominal}}
=
0.68(50)
=
34\text{ kN/mm},
$$

subject to multiplicative log-normal noise.

Each sensor reading multiplies the previous posterior by a likelihood concentrated near the stiffness values that could have produced that reading.

Consequently:

- Measurements below the healthy nominal stiffness reduce the probability of values near \(\theta=1\).
- The posterior shifts from the optimistic prior toward the damaged state.
- The posterior mean and MAP estimate move toward \(0.68\).
- The posterior density becomes narrower after additional measurements.
- The posterior standard deviation decreases.

Using the fixed random seed in the code and defining confidence as

$$
\left|
\widehat{\theta}_{Bayes}^{(k)}
-
0.68
\right|
<0.03
$$

together with

$$
\operatorname{SD}
\left(
\Theta\mid\mathbf{Y}^{(k)}
\right)
<0.05,
$$

the criterion is first reached at approximately

$$
\boxed{k=5}
$$

sensor readings.

The exact number can change for another random sequence because the sensor measurements are noisy.

The code therefore calculates the confidence step automatically instead of assuming a fixed value.

---

## Meaning of the Narrowing Posterior

As more measurements are observed, the likelihood information accumulates and the posterior density becomes narrower.

This means that the monitoring system is becoming more certain about the true remaining stiffness.

A narrow posterior centered near

$$
\theta=0.68
$$

indicates strong evidence that the component is operating at approximately \(68\%\) of its original stiffness.

This is important for structural safety decisions.

For example, suppose maintenance is required when

$$
\theta<0.70.
$$

A broad posterior may still assign considerable probability to both sides of the threshold, making the decision uncertain.

A narrow posterior lying mainly below \(0.70\) gives stronger evidence that the structure has crossed the safety threshold.

The probability of crossing a threshold \(\theta_c\) can be calculated as

$$
P(\Theta<\theta_c\mid\mathbf{Y}^{(k)})
=
\int_0^{\theta_c}
f_k(\theta)\,d\theta.
$$

Therefore,

$$
\boxed{
\text{Narrowing posterior}
\Rightarrow
\text{greater confidence in damage assessment}
}
$$

and

$$
\boxed{
\text{more reliable maintenance and safety decisions}.
}
$$

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

# Gaussian Mixture Clustering as Conditional Updating

Consider observations

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_n\in\mathbb{R}^d.
$$

We assume that each observation belongs to one of \(K\) latent clusters.

Let

$$
C_i\in\{1,2,\ldots,K\}
$$

be the unknown cluster label of observation \(\mathbf{x}_i\).

The prior probability of cluster \(k\) is

$$
P(C_i=k)=\phi_k,
$$

where

$$
\phi_k\geq 0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
$$

Conditional on \(C_i=k\), the observation follows a multivariate Gaussian distribution:

$$
\mathbf{X}_i\mid C_i=k
\sim
N(\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k).
$$

The multivariate Gaussian density is

$$
\mathcal{N}
\left(
\mathbf{x}\mid\boldsymbol{\mu},\boldsymbol{\Sigma}
\right)
=
\frac{1}
{
(2\pi)^{d/2}
|\boldsymbol{\Sigma}|^{1/2}
}
\exp
\left[
-\frac{1}{2}
(\mathbf{x}-\boldsymbol{\mu})^T
\boldsymbol{\Sigma}^{-1}
(\mathbf{x}-\boldsymbol{\mu})
\right].
$$

---

# 1. Marginal Density of an Observation

Using the law of total probability,

$$
p(\mathbf{x}_i)
=
\sum_{k=1}^{K}
p(\mathbf{x}_i,C_i=k).
$$

Using

$$
p(\mathbf{x}_i,C_i=k)
=
p(\mathbf{x}_i\mid C_i=k)P(C_i=k),
$$

we obtain

$$
p(\mathbf{x}_i)
=
\sum_{k=1}^{K}
p(\mathbf{x}_i\mid C_i=k)P(C_i=k).
$$

Substituting the Gaussian conditional density and cluster prior gives

$$
\boxed{
p(\mathbf{x}_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
}
$$

This is called a Gaussian mixture density because it is a weighted mixture of \(K\) Gaussian densities.

Each component represents one possible cluster, and \(\phi_k\) controls the contribution of component \(k\) to the overall density.

---

# 2. Posterior Cluster Probability

For a fixed observation \(\mathbf{x}_i\), Bayes' theorem gives

$$
P(C_i=k\mid \mathbf{X}_i=\mathbf{x}_i)
=
\frac{
p(\mathbf{x}_i\mid C_i=k)P(C_i=k)
}{
p(\mathbf{x}_i)
}.
$$

Using the Gaussian likelihood and cluster prior,

$$
P(C_i=k\mid \mathbf{X}_i=\mathbf{x}_i)
=
\frac{
\phi_k
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
}{
\displaystyle
\sum_{j=1}^{K}
\phi_j
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_j,
\boldsymbol{\Sigma}_j
\right)
}.
$$

Define

$$
\boxed{
\gamma_{ik}
=
P(C_i=k\mid \mathbf{X}_i=\mathbf{x}_i)
}
$$

so that

$$
\boxed{
\gamma_{ik}
=
\frac{
\phi_k
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
}{
\displaystyle
\sum_{j=1}^{K}
\phi_j
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_j,
\boldsymbol{\Sigma}_j
\right)
}
}
$$

The value \(\gamma_{ik}\) is called the responsibility of cluster \(k\) for observation \(i\).

It is a posterior probability because it combines:

1. The prior probability of cluster \(k\), \(\phi_k\).
2. The compatibility of \(\mathbf{x}_i\) with cluster \(k\), measured by the Gaussian density.
3. A normalizing denominator so that

$$
\sum_{k=1}^{K}\gamma_{ik}=1.
$$

---

# 3. One-Hot Encoding of the Latent Cluster Variable

Define the one-hot vector

$$
\mathbf{Z}_i
=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$

where

$$
Z_{ik}
=
\begin{cases}
1, & C_i=k,\\
0, & C_i\neq k.
\end{cases}
$$

Because \(Z_{ik}\) is an indicator variable,

$$
\mathbb{E}
\left[
Z_{ik}\mid\mathbf{X}_i=\mathbf{x}_i
\right]
=
1\cdot
P(C_i=k\mid\mathbf{x}_i)
+
0\cdot
P(C_i\neq k\mid\mathbf{x}_i).
$$

Therefore,

$$
\boxed{
\mathbb{E}
\left[
Z_{ik}\mid\mathbf{X}_i=\mathbf{x}_i
\right]
=
P(C_i=k\mid\mathbf{X}_i=\mathbf{x}_i)
=
\gamma_{ik}
}
$$

Hence,

$$
\boxed{
\mathbb{E}
\left[
\mathbf{Z}_i\mid\mathbf{X}_i=\mathbf{x}_i
\right]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
}
$$

Thus, the soft cluster assignment vector is exactly the conditional expectation of the latent one-hot cluster vector.

---

# 4. Soft and Hard Clustering

The vector

$$
\mathbb{E}
\left[
\mathbf{Z}_i\mid\mathbf{X}_i=\mathbf{x}_i
\right]
=
(\gamma_{i1},\gamma_{i2},\ldots,\gamma_{iK})^T
$$

provides a soft cluster assignment.

For example,

$$
(\gamma_{i1},\gamma_{i2},\gamma_{i3})
=
(0.10,0.65,0.25)
$$

means that the point has:

- \(10\%\) posterior probability of belonging to cluster 1.
- \(65\%\) posterior probability of belonging to cluster 2.
- \(25\%\) posterior probability of belonging to cluster 3.

A hard assignment chooses only the cluster with the largest posterior probability:

$$
\boxed{
\widehat{C}_i
=
\underset{1\leq k\leq K}
{\operatorname{argmax}}
\;
\gamma_{ik}
}
$$

Therefore:

- Soft clustering preserves uncertainty about cluster membership.
- Hard clustering assigns each observation to exactly one cluster.
- Hard clustering loses information about competing cluster probabilities.

---

# 5. Conditional Expectation Given the Cluster

For cluster \(k\),

$$
\mathbf{X}_i\mid C_i=k
\sim
N(\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k).
$$

The expected value of a multivariate Gaussian distribution is its mean vector. Therefore,

$$
\boxed{
\mathbb{E}
\left[
\mathbf{X}_i\mid C_i=k
\right]
=
\boldsymbol{\mu}_k
}
$$

Thus, \(\boldsymbol{\mu}_k\) represents the center or typical location of cluster \(k\).

The two important conditional expectations have different meanings:

$$
\mathbb{E}
\left[
\mathbf{Z}_i\mid\mathbf{X}_i=\mathbf{x}_i
\right]
$$

gives the posterior cluster-membership probabilities of an observed data point.

In contrast,

$$
\mathbb{E}
\left[
\mathbf{X}_i\mid C_i=k
\right]
=
\boldsymbol{\mu}_k
$$

gives the mean location of cluster \(k\) in the feature space.

# 6. Complete-Data Likelihood

If the latent indicators \(z_{ik}\) were observed, the complete-data likelihood would be

$$
p(\mathbf{x}_1,\ldots,\mathbf{x}_n,
\mathbf{z}_1,\ldots,\mathbf{z}_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
\right]^{z_{ik}}.
$$

Taking logarithms,

$$
\ell_c
=
\log p(\mathbf{X},\mathbf{Z}).
$$

Using the logarithm of a product,

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
\right]
}
$$

Expanding the Gaussian log-density gives

$$
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
-\frac{d}{2}\log(2\pi)
-\frac{1}{2}\log|\boldsymbol{\Sigma}_k|
\right.
$$

$$
\left.
-\frac{1}{2}
(\mathbf{x}_i-\boldsymbol{\mu}_k)^T
\boldsymbol{\Sigma}_k^{-1}
(\mathbf{x}_i-\boldsymbol{\mu}_k)
\right].
$$

If \(z_{ik}\) were known, observations belonging to each component would be known.

The likelihood would then separate into \(K\) ordinary Gaussian estimation problems, making it easy to calculate:

- The cluster proportions.
- The cluster means.
- The cluster covariance matrices.

The difficulty is that the latent indicators are not observed.

---

# 7. EM Interpretation

The Expectation-Maximization algorithm replaces the unknown indicator \(z_{ik}\) with its conditional expectation:

$$
z_{ik}
\longrightarrow
\mathbb{E}
\left[
Z_{ik}
\mid
\mathbf{X}_i=\mathbf{x}_i
\right].
$$

Therefore,

$$
z_{ik}
\longrightarrow
\gamma_{ik}.
$$

Using the current parameter estimates, the E-step calculates

$$
\boxed{
\gamma_{ik}
=
\frac{
\phi_k
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
}{
\displaystyle
\sum_{j=1}^{K}
\phi_j
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_j,
\boldsymbol{\Sigma}_j
\right)
}
}
$$

The expected complete-data log-likelihood is

$$
Q
=
\mathbb{E}
\left[
\ell_c
\mid
\mathbf{X}
\right].
$$

Therefore,

$$
\boxed{
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
\right]
}
$$

The E-step is a conditional update because it changes the prior cluster probabilities into posterior membership probabilities after observing each data point.

Symbolically,

$$
\text{Prior}
\times
\text{Likelihood}
\longrightarrow
\text{Posterior responsibility}.
$$

That is,

$$
\phi_k
\times
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k
\right)
\longrightarrow
\gamma_{ik}.
$$

---

# 8. Parameter Updates in the M-Step

Define the effective number of observations assigned to cluster \(k\) as

$$
\boxed{
N_k
=
\sum_{i=1}^{n}\gamma_{ik}
}
$$

This is not necessarily an integer because the responsibilities are fractional probabilities.

---

## Mixture-Weight Update

Maximizing

$$
\sum_{k=1}^{K}N_k\log\phi_k
$$

subject to

$$
\sum_{k=1}^{K}\phi_k=1
$$

gives

$$
\boxed{
\phi_k^{\text{new}}
=
\frac{N_k}{n}
}
$$

Thus, the new mixture weight is the effective fraction of observations assigned to cluster \(k\).

---

## Mean Update

Differentiating \(Q\) with respect to \(\boldsymbol{\mu}_k\) and setting the result equal to zero gives

$$
\sum_{i=1}^{n}
\gamma_{ik}
\boldsymbol{\Sigma}_k^{-1}
(\mathbf{x}_i-\boldsymbol{\mu}_k)
=
0.
$$

Therefore,

$$
\boxed{
\boldsymbol{\mu}_k^{\text{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}\mathbf{x}_i
}
$$

This is a responsibility-weighted average.

---

## Covariance Update

The covariance update is

$$
\boxed{
\boldsymbol{\Sigma}_k^{\text{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
\left(
\mathbf{x}_i-\boldsymbol{\mu}_k^{\text{new}}
\right)
\left(
\mathbf{x}_i-\boldsymbol{\mu}_k^{\text{new}}
\right)^T
}
$$

Therefore, each observation contributes to every component in proportion to its responsibility.

For example, if

$$
\gamma_{ik}=0.80,
$$

then observation \(i\) contributes \(80\%\) of its weight to the parameter updates for component \(k\).

Thus, responsibilities act as fractional cluster-membership weights.

---

# 9. Gaussian Mixture Clustering as Conditional Updating

Gaussian mixture clustering can be understood as a repeated process of conditional probability updating. The mixture weight \(\phi_k\) is the prior probability that an observation belongs to cluster \(k\). The Gaussian density

$$
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right)
$$

measures how compatible the observed point is with that cluster. Bayes' theorem combines the prior and likelihood to calculate the posterior responsibility

$$
\gamma_{ik}
=
P(C_i=k\mid\mathbf{X}_i=\mathbf{x}_i).
$$

The vector

$$
\mathbb{E}
\left[
\mathbf{Z}_i
\mid
\mathbf{X}_i=\mathbf{x}_i
\right]
=
(\gamma_{i1},\ldots,\gamma_{iK})^T
$$

is the soft cluster assignment. The M-step then updates the mixture weights, means and covariance matrices using these posterior probabilities as fractional weights. The E-step and M-step are repeated until the model parameters and responsibilities stop changing significantly. Therefore, GMM clustering is probabilistic clustering based on conditional expectations of latent membership variables.

In [9]:
from google.colab import files

# Upload either:
# 1. archive.zip
# 2. CC GENERAL.csv

uploaded = files.upload()

input_path = next(iter(uploaded.keys()))

print("Uploaded file:", input_path)

Saving archive.zip to archive (1).zip
Uploaded file: archive (1).zip


In [11]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    """
    Two-dimensional Gaussian Mixture financial segmenter.

    The default features are:
        PURCHASES
        CREDIT_LIMIT
    """

    def __init__(
        self,
        features=("PURCHASES", "CREDIT_LIMIT"),
        n_components=3,
        test_size=0.20,
        random_state=42
    ):
        self.features = list(features)
        self.n_components = n_components
        self.test_size = test_size
        self.random_state = random_state

        self.scaler = StandardScaler()

        self.gmm = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            n_init=10,
            max_iter=500,
            reg_covar=1e-6,
            random_state=random_state
        )

    @staticmethod
    def _resolve_csv(path):
        """
        Return the CSV path.

        If a ZIP file is supplied, extract it and locate
        the first CSV file inside it.
        """

        path = Path(path)

        if not path.exists():
            raise FileNotFoundError(
                f"File not found: {path}"
            )

        if path.suffix.lower() == ".csv":
            return path

        if path.suffix.lower() == ".zip":
            output_directory = (
                path.parent
                / f"{path.stem}_extracted"
            )

            output_directory.mkdir(
                parents=True,
                exist_ok=True
            )

            with zipfile.ZipFile(path, "r") as archive:
                archive.extractall(output_directory)

            csv_files = sorted(
                output_directory.rglob("*.csv")
            )

            if not csv_files:
                raise FileNotFoundError(
                    "No CSV file was found inside the ZIP archive."
                )

            return csv_files[0]

        raise ValueError(
            "Provide a CSV file or a ZIP archive containing a CSV file."
        )

    def load_and_prepare(self, path):
        """
        Load, clean, split and standardize the selected features.
        """

        csv_path = self._resolve_csv(path)

        self.data = pd.read_csv(csv_path)

        missing_columns = [
            column
            for column in self.features
            if column not in self.data.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )

        selected_data = (
            self.data[self.features]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        if len(selected_data) < self.n_components:
            raise ValueError(
                "There are too few complete observations "
                "for the requested number of components."
            )

        self.clean_data = selected_data

        (
            self.X_train_raw,
            self.X_test_raw
        ) = train_test_split(
            selected_data,
            test_size=self.test_size,
            random_state=self.random_state
        )

        # Fit the scaler only on the training set.
        self.X_train = self.scaler.fit_transform(
            self.X_train_raw
        )

        self.X_test = self.scaler.transform(
            self.X_test_raw
        )

        print("CSV file:", csv_path)
        print("Complete observations:", len(selected_data))
        print("Training observations:", len(self.X_train_raw))
        print("Test observations:", len(self.X_test_raw))

        return self

    def fit(self):
        """
        Fit the Gaussian mixture model using EM.
        """

        self.gmm.fit(self.X_train)

        self.train_labels = self.gmm.predict(
            self.X_train
        )

        self.test_labels = self.gmm.predict(
            self.X_test
        )

        self.train_responsibilities = (
            self.gmm.predict_proba(self.X_train)
        )

        self.test_responsibilities = (
            self.gmm.predict_proba(self.X_test)
        )

        self.train_score = self.gmm.score(
            self.X_train
        )

        self.test_score = self.gmm.score(
            self.X_test
        )

        print("\nGMM fitting results")
        print("-------------------")
        print("Converged:", self.gmm.converged_)
        print("Number of EM iterations:", self.gmm.n_iter_)

        print(
            "Average training log-likelihood:",
            round(self.train_score, 6)
        )

        print(
            "Average test log-likelihood:",
            round(self.test_score, 6)
        )

        return self

    def component_summary(self):
        """
        Return component weights and means in the original units.
        """

        means_original = self.scaler.inverse_transform(
            self.gmm.means_
        )

        summary = pd.DataFrame(
            means_original,
            columns=[
                f"Mean {feature}"
                for feature in self.features
            ]
        )

        summary.insert(
            0,
            "Component",
            np.arange(self.n_components)
        )

        summary["Mixture weight"] = (
            self.gmm.weights_
        )

        return summary

    def plot_density_heatmap(self):
        """
        Plot an empirical two-dimensional density heatmap
        of the raw training observations with marginals.
        """

        plot_data = (
            self.X_train_raw
            .reset_index(drop=True)
        )

        figure = px.density_heatmap(
            plot_data,
            x=self.features[0],
            y=self.features[1],
            nbinsx=60,
            nbinsy=60,
            marginal_x="histogram",
            marginal_y="histogram",
            title=(
                "Empirical 2D Density of the Raw "
                "Training Data"
            ),
            labels={
                self.features[0]: "Purchases",
                self.features[1]: "Credit limit"
            }
        )

        figure.update_layout(
            template="plotly_white"
        )

        figure.show()

        return figure

    def _responsibility_grid(
        self,
        grid_size=250
    ):
        """
        Evaluate responsibilities over a fine grid
        in the original feature units.
        """

        combined_data = pd.concat(
            [
                self.X_train_raw,
                self.X_test_raw
            ],
            axis=0
        )

        x_min = combined_data[
            self.features[0]
        ].min()

        x_max = combined_data[
            self.features[0]
        ].max()

        y_min = combined_data[
            self.features[1]
        ].min()

        y_max = combined_data[
            self.features[1]
        ].max()

        x_range = max(x_max - x_min, 1.0)
        y_range = max(y_max - y_min, 1.0)

        x_padding = 0.03 * x_range
        y_padding = 0.03 * y_range

        # These financial features cannot be negative.
        x_lower = max(0.0, x_min - x_padding)
        y_lower = max(0.0, y_min - y_padding)

        x_axis = np.linspace(
            x_lower,
            x_max + x_padding,
            grid_size
        )

        y_axis = np.linspace(
            y_lower,
            y_max + y_padding,
            grid_size
        )

        xx, yy = np.meshgrid(
            x_axis,
            y_axis
        )

        raw_grid = pd.DataFrame(
            {
                self.features[0]: xx.ravel(),
                self.features[1]: yy.ravel()
            }
        )

        scaled_grid = self.scaler.transform(
            raw_grid
        )

        responsibility_matrix = (
            self.gmm.predict_proba(scaled_grid)
        )

        maximum_responsibility = (
            responsibility_matrix
            .max(axis=1)
            .reshape(xx.shape)
        )

        hard_region = (
            responsibility_matrix
            .argmax(axis=1)
            .reshape(xx.shape)
        )

        return (
            x_axis,
            y_axis,
            maximum_responsibility,
            hard_region
        )

    def _plot_assignments(
        self,
        raw_data,
        labels,
        title
    ):
        """
        Plot observations over the continuous maximum-
        responsibility confidence surface.
        """

        (
            x_axis,
            y_axis,
            maximum_responsibility,
            hard_region
        ) = self._responsibility_grid()

        figure = go.Figure()

        figure.add_trace(
            go.Contour(
                x=x_axis,
                y=y_axis,
                z=maximum_responsibility,
                contours=dict(
                    start=1 / self.n_components,
                    end=1.0,
                    size=0.03,
                    coloring="heatmap"
                ),
                line=dict(width=0),
                opacity=0.65,
                colorbar=dict(
                    title="Maximum<br>responsibility"
                ),
                hovertemplate=(
                    "Purchases=%{x:.2f}<br>"
                    "Credit limit=%{y:.2f}<br>"
                    "Maximum responsibility=%{z:.3f}"
                    "<extra></extra>"
                ),
                name="Responsibility confidence"
            )
        )

        raw_values = raw_data.to_numpy()

        for component in range(
            self.n_components
        ):
            component_mask = (
                labels == component
            )

            figure.add_trace(
                go.Scatter(
                    x=raw_values[
                        component_mask,
                        0
                    ],
                    y=raw_values[
                        component_mask,
                        1
                    ],
                    mode="markers",
                    name=f"Component {component}",
                    marker=dict(
                        size=6,
                        opacity=0.75
                    ),
                    hovertemplate=(
                        "Purchases=%{x:.2f}<br>"
                        "Credit limit=%{y:.2f}<br>"
                        f"Hard component={component}"
                        "<extra></extra>"
                    )
                )
            )

        figure.update_layout(
            title=title,
            xaxis_title=self.features[0],
            yaxis_title=self.features[1],
            template="plotly_white"
        )

        figure.show()

        return figure

    def plot_training_assignments(self):
        """
        Plot the training observations and their assignments.
        """

        return self._plot_assignments(
            self.X_train_raw,
            self.train_labels,
            (
                "Training Assignments over the "
                "Maximum-Responsibility Surface"
            )
        )

    def plot_test_assignments(self):
        """
        Plot out-of-sample test observations and assignments.
        """

        return self._plot_assignments(
            self.X_test_raw,
            self.test_labels,
            (
                "Out-of-Sample Test Assignments over the "
                "Maximum-Responsibility Surface"
            )
        )

In [12]:
segmenter = GMMFinancialSegmenter(
    features=("PURCHASES", "CREDIT_LIMIT"),
    n_components=3,
    test_size=0.20,
    random_state=42
)

segmenter.load_and_prepare(
    input_path
)

segmenter.fit()

print("\nComponent summary in original units")
display(
    segmenter
    .component_summary()
    .round(3)
)

# Figure 1: Empirical raw-data density
segmenter.plot_density_heatmap()

# Figure 2: Training assignments
segmenter.plot_training_assignments()

# Figure 3: Out-of-sample test assignments
segmenter.plot_test_assignments()

CSV file: archive (1)_extracted/CC GENERAL.csv
Complete observations: 8949
Training observations: 7159
Test observations: 1790

GMM fitting results
-------------------
Converged: True
Number of EM iterations: 20
Average training log-likelihood: -1.606159
Average test log-likelihood: -1.688831

Component summary in original units


,Component,Mean PURCHASES,Mean CREDIT_LIMIT,Mixture weight
0,0,4631.641,9578.342,0.104
1,1,183.625,2070.208,0.443
2,2,918.776,5744.381,0.453


# 10. Computational Simulation and Out-of-Sample Validation

The implementation performs the following steps:

1. It extracts the dataset if a ZIP archive is supplied.
2. It selects the two features

$$
\text{PURCHASES}
\quad\text{and}\quad
\text{CREDIT\_LIMIT}.
$$

3. Rows containing missing values in these features are removed.
4. The observations are divided into:

$$
80\% \text{ training data}
$$

and

$$
20\% \text{ test data}.
$$

5. The standardization parameters are estimated using only the training data.
6. A three-component full-covariance Gaussian mixture model is fitted using the EM algorithm.
7. The model reports whether EM converged and the number of iterations required.
8. The average test log-likelihood is calculated using

$$
\frac{1}{n_{\text{test}}}
\sum_{i\in\text{test}}
\log
p(\mathbf{x}_i).
$$

A larger average test log-likelihood indicates that the fitted density assigns greater probability to unseen observations.

With the supplied credit-card dataset and random seed \(42\), a typical run produces approximately:

$$
\text{EM iterations}=20
$$

and

$$
\text{Average test log-likelihood}
\approx -1.689.
$$

Small numerical differences may occur between software versions.

---

## Interpretation of the Component Means

A typical fitted model produces component centers approximately similar to:

| Component | Mean PURCHASES | Mean CREDIT_LIMIT | Mixture weight |
|---:|---:|---:|---:|
| 0 | \(4632\) | \(9578\) | \(0.104\) |
| 1 | \(184\) | \(2070\) | \(0.443\) |
| 2 | \(919\) | \(5744\) | \(0.453\) |

The component numbers are arbitrary labels and may be permuted in another run.

A possible interpretation is:

- One relatively small group with high purchases and high credit limits.
- One large group with low purchases and relatively low credit limits.
- One large group with moderate purchases and medium-to-high credit limits.

These interpretations are based only on the two selected variables and should not automatically be treated as complete customer profiles.

---

## Interpretation of the Density Heatmap

The first figure displays the empirical concentration of the raw training observations.

The marginal histograms show the individual distributions of purchases and credit limit.

The heatmap helps identify:

- Dense regions containing many customers.
- Sparse regions.
- Possible multimodal structure.
- Outliers with unusually high purchases or credit limits.

---

## Interpretation of the Responsibility Contours

At every coordinate-grid point, the model calculates the complete responsibility vector

$$
\mathbb{E}
\left[
\mathbf{Z}
\mid
\mathbf{X}=\mathbf{x}_{\text{grid}}
\right]
=
\begin{bmatrix}
\gamma_1(\mathbf{x}_{\text{grid}})\\
\gamma_2(\mathbf{x}_{\text{grid}})\\
\gamma_3(\mathbf{x}_{\text{grid}})
\end{bmatrix}.
$$

The background contour displays

$$
\max_k
\gamma_k(\mathbf{x}_{\text{grid}}).
$$

Therefore:

- Values close to \(1\) indicate confident cluster membership.
- Lower values indicate that two or more clusters have similar posterior probabilities.
- Low-confidence areas represent regions of cluster ambiguity.
- The colored data points show the hard assignment

$$
\widehat{C}_i
=
\arg\max_k\gamma_{ik}.
$$

The continuous background is therefore a visual summary of the soft conditional expectation vector derived theoretically.

---

## Training and Test Assignment Plots

The training assignment plot shows how the fitted GMM divides the observations used during parameter estimation.

The test assignment plot uses exactly the same fitted model, scaler and responsibility surface, but overlays previously unseen observations.

If test observations follow similar regions and confidence patterns to the training observations, this suggests that the fitted mixture density generalizes reasonably well.

Observations near low-confidence boundaries are uncertain because their Gaussian likelihoods under two or more components are similar.

---

# Final Conclusion

Gaussian mixture clustering is not merely a deterministic division of points.

For every observation, the model first calculates the prior-weighted compatibility

$$
\phi_k
\mathcal{N}
\left(
\mathbf{x}_i
\mid
\boldsymbol{\mu}_k,
\boldsymbol{\Sigma}_k
\right).
$$

It then normalizes these values to obtain the posterior responsibility

$$
\gamma_{ik}
=
P(C_i=k\mid\mathbf{X}_i=\mathbf{x}_i).
$$

Thus,

$$
\boxed{
\text{Soft assignment}
=
\mathbb{E}
\left[
\mathbf{Z}_i
\mid
\mathbf{X}_i=\mathbf{x}_i
\right]
}
$$

and the EM algorithm repeatedly alternates between:

$$
\boxed{
\text{E-step: update conditional membership probabilities}
}
$$

and

$$
\boxed{
\text{M-step: update cluster parameters using those probabilities}.
}
$$

Therefore, Gaussian mixture clustering is a repeated conditional-updating procedure for latent cluster membership.